# Kriteria 1 Advanced - Persiapan Data dan Baseline LSTM

Notebook referensi ini sekarang ditingkatkan sampai **Kriteria 1 tahap Advanced**.

Yang harus tetap terpenuhi dari tahap sebelumnya:
- memakai minimal 3 fitur input,
- membuat heatmap korelasi,
- membagi data menjadi `train`, `validation`, dan `test`,
- melakukan scaling tanpa data leakage,
- membuat pipeline data dengan `tf.data.Dataset`,
- membuat analisis dekomposisi target `Close`,
- membangun baseline LSTM dan melatihnya dengan `model.fit()`.

Tambahan khusus tahap advanced:
- menentukan `window size` berdasarkan analisis lag dengan **ACF** dan **PACF**,
- menambahkan **feature engineering** berbasis **rolling statistic**.

Notebook ini tetap fokus pada **Kriteria 1**, jadi belum masuk ke custom model atau custom training loop.


In [ ]:
import importlib.util
import subprocess
import sys

PACKAGE_MAP = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'scikit-learn': 'sklearn',
    'statsmodels': 'statsmodels',
    'tensorflow': 'tensorflow',
}

missing_packages = [pkg for pkg, module_name in PACKAGE_MAP.items() if importlib.util.find_spec(module_name) is None]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
    print('Installed missing packages:', missing_packages)
else:
    print('All required packages are already available.')

import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, pacf

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

SEED = 42
CSV_URL = 'https://drive.google.com/uc?export=download&id=1hpsqSpfjdqIZWqwd259klQSeaNSe5Trr'
ROLLING_WINDOW = 24
HORIZON = 24
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
BATCH_SIZE = 64
EPOCHS = 12
WINDOW_CANDIDATES = (24, 48, 72, 96, 168)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)
print('TensorFlow version:', tf.__version__)


## 1. Load Dataset, Tambahkan Rolling Feature, dan Analisis Awal

Dataset memiliki kolom:
- `Date`
- `Close`
- `Volume USDT`
- `RSI`
- `MACD_Hist`
- `ATR`
- `KAMAO`

Di tahap advanced, kita menambahkan dua fitur rolling yang bersifat kausal:
- `close_roll_mean_24`
- `close_roll_std_24`

Keduanya dihitung dari riwayat sebelumnya pada seri `Close`, sehingga tetap aman dari data leakage.


In [ ]:
BASE_FEATURE_COLUMNS = ['Close', 'Volume USDT', 'RSI', 'MACD_Hist', 'ATR', 'KAMAO']
TARGET_COLUMN = 'Close'

def load_dataset(csv_url=CSV_URL):
    df = pd.read_csv(csv_url)
    df.columns = [col.strip() for col in df.columns]
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date']).sort_values('Date').drop_duplicates(subset=['Date']).reset_index(drop=True)
    df[BASE_FEATURE_COLUMNS] = df[BASE_FEATURE_COLUMNS].apply(pd.to_numeric, errors='coerce')
    df = df.dropna(subset=BASE_FEATURE_COLUMNS).reset_index(drop=True)
    return df[['Date'] + BASE_FEATURE_COLUMNS].copy()

def add_rolling_features(df, target_column=TARGET_COLUMN, rolling_window=ROLLING_WINDOW):
    enriched = df.copy()
    enriched['close_roll_mean_24'] = enriched[target_column].rolling(rolling_window, min_periods=rolling_window).mean()
    enriched['close_roll_std_24'] = enriched[target_column].rolling(rolling_window, min_periods=rolling_window).std()
    enriched = enriched.dropna().reset_index(drop=True)
    return enriched

def chronological_split(df, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO):
    total_ratio = train_ratio + val_ratio + test_ratio
    if not np.isclose(total_ratio, 1.0):
        raise ValueError('Split ratio harus berjumlah 1.0')
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)
    train_df = df.iloc[:train_end].reset_index(drop=True)
    val_df = df.iloc[train_end:val_end].reset_index(drop=True)
    test_df = df.iloc[val_end:].reset_index(drop=True)
    return train_df, val_df, test_df

def fit_scalers(train_df, feature_columns):
    scalers = {}
    for col in feature_columns:
        scaler = MinMaxScaler()
        scaler.fit(train_df[[col]])
        scalers[col] = scaler
    return scalers

def transform_with_scalers(df, feature_columns, scalers):
    transformed = df[['Date']].copy()
    for col in feature_columns:
        transformed[col] = scalers[col].transform(df[[col]]).astype(np.float32)
    return transformed

def select_window_size_from_lag_analysis(series, candidate_windows=WINDOW_CANDIDATES):
    max_lag = max(candidate_windows)
    acf_values = acf(series, nlags=max_lag, fft=True)
    pacf_values = pacf(series, nlags=max_lag, method='ywm')
    significance_threshold = 1.96 / np.sqrt(len(series))

    lag_rows = []
    selected_window = 72
    for lag in candidate_windows:
        acf_at_lag = float(acf_values[lag])
        pacf_at_lag = float(pacf_values[lag])
        score = max(abs(acf_at_lag), abs(pacf_at_lag))
        lag_rows.append({
            'lag': lag,
            'acf': acf_at_lag,
            'pacf': pacf_at_lag,
            'score': score,
            'significant': score >= significance_threshold,
        })
        if score >= significance_threshold and selected_window == 72:
            selected_window = lag

    lag_summary = pd.DataFrame(lag_rows)
    return selected_window, significance_threshold, lag_summary

def make_supervised_windows(df, feature_columns, target_column, window_size, horizon=HORIZON):
    feature_values = df[feature_columns].to_numpy(dtype=np.float32)
    target_values = df[target_column].to_numpy(dtype=np.float32)
    timestamps = df['Date'].to_numpy()

    X, y, y_timestamps = [], [], []
    for end_idx in range(window_size, len(df) - horizon + 1):
        start_idx = end_idx - window_size
        target_slice = slice(end_idx, end_idx + horizon)
        X.append(feature_values[start_idx:end_idx])
        y.append(target_values[target_slice].reshape(horizon, 1))
        y_timestamps.append(timestamps[target_slice])

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    y_timestamps = np.asarray(y_timestamps)
    return X, y, y_timestamps

def make_tf_dataset(X, y, batch_size=BATCH_SIZE, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y.astype(np.float32)))
    if shuffle:
        dataset = dataset.shuffle(min(len(X), 2048), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

raw_df = load_dataset(CSV_URL)
feature_engineered_df = add_rolling_features(raw_df)
FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + ['close_roll_mean_24', 'close_roll_std_24']

train_raw_df, val_raw_df, test_raw_df = chronological_split(feature_engineered_df)

print('Shape raw data after rolling feature engineering:', feature_engineered_df.shape)
print('Feature columns:', FEATURE_COLUMNS)
print('Train / Val / Test:', len(train_raw_df), len(val_raw_df), len(test_raw_df))
display(train_raw_df.head())

plt.figure(figsize=(12, 8))
sns.heatmap(train_raw_df[FEATURE_COLUMNS].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Heatmap Korelasi Fitur pada Data Train')
plt.show()

decomposition = seasonal_decompose(train_raw_df[TARGET_COLUMN], model='additive', period=24, extrapolate_trend='freq')
decomposition.plot()
plt.suptitle('Dekomposisi Target Close pada Data Train', y=1.02)
plt.show()

selected_window_size, significance_threshold, lag_summary_df = select_window_size_from_lag_analysis(train_raw_df[TARGET_COLUMN])
display(lag_summary_df)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
plot_acf(train_raw_df[TARGET_COLUMN], lags=max(WINDOW_CANDIDATES), ax=axes[0])
plot_pacf(train_raw_df[TARGET_COLUMN], lags=max(WINDOW_CANDIDATES), ax=axes[1], method='ywm')
axes[0].set_title('ACF Close pada Data Train')
axes[1].set_title('PACF Close pada Data Train')
plt.tight_layout()
plt.show()

print(f'Significance threshold: {significance_threshold:.5f}')
print(f'Selected window size from ACF/PACF analysis: {selected_window_size}')

scalers = fit_scalers(train_raw_df, FEATURE_COLUMNS)
train_scaled_df = transform_with_scalers(train_raw_df, FEATURE_COLUMNS, scalers)
val_scaled_df = transform_with_scalers(val_raw_df, FEATURE_COLUMNS, scalers)
test_scaled_df = transform_with_scalers(test_raw_df, FEATURE_COLUMNS, scalers)

X_train, y_train, train_timestamps = make_supervised_windows(train_scaled_df, FEATURE_COLUMNS, TARGET_COLUMN, selected_window_size)
X_val, y_val, val_timestamps = make_supervised_windows(val_scaled_df, FEATURE_COLUMNS, TARGET_COLUMN, selected_window_size)
X_test, y_test, test_timestamps = make_supervised_windows(test_scaled_df, FEATURE_COLUMNS, TARGET_COLUMN, selected_window_size)

train_ds = make_tf_dataset(X_train, y_train, shuffle=True)
val_ds = make_tf_dataset(X_val, y_val, shuffle=False)
test_ds = make_tf_dataset(X_test, y_test, shuffle=False)

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('X_val shape:', X_val.shape)
print('X_test shape:', X_test.shape)

sample_batch_x, sample_batch_y = next(iter(train_ds))
print('Sample batch from tf.data.Dataset:', sample_batch_x.shape, sample_batch_y.shape)


## 2. Bangun Baseline LSTM dan Latih dengan `model.fit()`

Modelnya tetap baseline LSTM sederhana karena fokus kita masih Kriteria 1. Perbedaannya sekarang:
- input model sudah memakai fitur rolling,
- `window size` tidak lagi hardcoded, tetapi dipilih dari hasil analisis `ACF` dan `PACF`,
- training memakai pipeline `tf.data.Dataset`.


In [ ]:
baseline_lstm_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(selected_window_size, len(FEATURE_COLUMNS))),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(HORIZON),
    tf.keras.layers.Reshape((HORIZON, 1)),
], name='baseline_lstm_advanced')

baseline_lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mae',
    metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae')],
)

baseline_lstm_model.summary()

history = baseline_lstm_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=1,
)

test_loss, test_mae = baseline_lstm_model.evaluate(test_ds, verbose=0)
print(f'Test loss: {test_loss:.6f}')
print(f'Test MAE : {test_mae:.6f}')

plt.figure(figsize=(12, 5))
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Training Curve Baseline LSTM')
plt.xlabel('Epoch')
plt.ylabel('MAE Loss')
plt.legend()
plt.show()

sample_prediction = baseline_lstm_model.predict(X_test[:1], verbose=0)[0, :, 0]
sample_actual = y_test[0, :, 0]

comparison_df = pd.DataFrame({
    'step': np.arange(1, HORIZON + 1),
    'actual_close_scaled': sample_actual,
    'predicted_close_scaled': sample_prediction,
})
comparison_df['difference'] = comparison_df['predicted_close_scaled'] - comparison_df['actual_close_scaled']
display(comparison_df)

plt.figure(figsize=(14, 5))
plt.plot(comparison_df['step'], comparison_df['actual_close_scaled'], marker='o', label='Actual')
plt.plot(comparison_df['step'], comparison_df['predicted_close_scaled'], marker='x', label='Predicted')
plt.title('Contoh Prediksi 24 Step dari Baseline LSTM')
plt.xlabel('Forecast Step')
plt.ylabel('Scaled Close')
plt.legend()
plt.show()


## Kesimpulan Tahap Ini

Kalau notebook ini berhasil dijalankan tanpa error, maka poin-poin **Kriteria 1 Advanced** yang sudah terpenuhi adalah:
- memakai lebih dari 3 fitur input,
- membuat heatmap korelasi,
- membagi data menjadi `train`, `validation`, dan `test`,
- melakukan scaling tanpa data leakage,
- membuat pipeline data dengan `tf.data.Dataset`,
- membuat analisis dekomposisi target `Close`,
- menentukan `window size` dari analisis lag `ACF` dan `PACF`,
- menambahkan feature engineering rolling statistic,
- membangun baseline LSTM,
- melatih baseline LSTM dengan `model.fit()`.

Tahap berikutnya setelah ini adalah mulai masuk ke **Kriteria 2**, yaitu membangun arsitektur model kustom seperti Seq2Seq LSTM Teacher Forcing dan custom layer.
